# Follow-up experiments (results are pushed to GitHub after every stage)

Run on a **GPU runtime**; *Runtime → Run all* works. Each stage saves to Drive and then pushes its reports to `saved_results/followup_<stage>/` on your fork, so Claude can pull and check them while you work on the thesis. Every stage can be re-run: finished work is reused.

| stage | what | time (GPU) |
|---|---|---|
| A | setup, push check, tests | 5 min |
| B | decoding the read word from EEG (English band power, Persian) + sentiment of the decoded text | ~15 min |
| C | English vs Persian EEG directions in a shared text space | ~10 min |
| D | probe of the raw ZuCo files for fixation-locked EEG | 2 min |
| E | extract fixation-related potentials (all readers; resumable) | 1–3 h |
| F | fixation-related potentials: timing check, N400 regression, signal probes, encoding scan with surprisal control, decoding | ~1 h |
| G | brain-tuning the LoRA LLM with within-sentence EEG targets (1 fold) | ~1 h |
| H | optional: all 5 folds of the LLM with EEG tokens | many hours |

## A. Setup

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/MohammadJRanjbar/zuco-multimodal-sentiment.git'
BRANCH = 'feature/neurolm-eeg-probe'
REPO_DIR = '/content/zuco-multimodal-sentiment'
if not os.path.exists(REPO_DIR):
    !git clone --branch $BRANCH $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR fetch origin && git -C $REPO_DIR checkout $BRANCH && git -C $REPO_DIR -c user.name=colab -c user.email=colab@users.noreply.github.com pull --rebase origin $BRANCH
%cd $REPO_DIR
!pip install -q -r requirements-diagnostics.txt
!git log --oneline -1

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
THESIS_ROOT = '/content/drive/MyDrive/Thesis'
ARTIFACTS = f'{THESIS_ROOT}/CachedArtifacts/zuco_multimodal_sentiment'
MAT_DIR = f'{THESIS_ROOT}/Data/zuco_og_raw'
LABELS_CSV = f'{THESIS_ROOT}/Data/zuco_sentiment_labels_task1_fixed.csv'
WORD_EEG_DIR = f'{ARTIFACTS}/word_eeg/TRT'
FRP_DIR = f'{ARTIFACTS}/word_eeg/FRP'
TECO_TRT_DIR = f'{THESIS_ROOT}/Data/TRT_Total'
TECO_LABELS = f'{THESIS_ROOT}/Data/teco_sentiment_labels_task1.csv'
RESULTS = f'{THESIS_ROOT}/Results/zuco_multimodal_sentiment/followup'
LORA_RESULTS = f'{THESIS_ROOT}/Results/zuco_multimodal_sentiment/eeg_text_lora'
VECTOR_CACHE = '/content/vector_cache'
os.environ['HF_HOME'] = f'{ARTIFACTS}/hf_cache'
os.makedirs(RESULTS, exist_ok=True)
DEVICE = 'cuda' if os.system('nvidia-smi > /dev/null 2>&1') == 0 else 'cpu'
for path in [MAT_DIR, LABELS_CSV, WORD_EEG_DIR, TECO_TRT_DIR, TECO_LABELS]:
    print('ok ' if os.path.exists(path) else 'MISSING', path)
print('device:', DEVICE)

# GitHub push: the GITHUB_TOKEN secret (fine-grained, Contents: Read and write on your fork).
os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
GIT_NAME, GIT_EMAIL = 'mohammadranjbar', 'mjrkalhar@gmail.com'
token = os.environ['GITHUB_TOKEN']
check = subprocess.run(['git', 'push', '--dry-run', REPO_URL.replace('https://', f'https://x-access-token:{token}@'),
                        f'HEAD:refs/heads/push-check-{os.getpid()}'], capture_output=True, text=True)
message = (check.stdout + check.stderr).replace(token, '***')
if check.returncode == 0:
    print('push check: OK, results will be pushed to GitHub')
else:
    print('push check FAILED; results will stay on Drive only. Fix the token (see the thesis notes).\n', message[-800:])

def push(stage, *sources):
    """Commit the small result files of one stage to saved_results/followup_<stage>/ and push."""
    sources = [s for s in sources if os.path.exists(s)]
    if not sources:
        print('nothing to push for', stage); return
    result = subprocess.run(['python', 'scripts/save_results_to_github.py', '--name', f'followup_{stage}', '--source',
                             *sources, '--push', '--author-name', GIT_NAME, '--author-email', GIT_EMAIL],
                            capture_output=True, text=True)
    print((result.stdout + result.stderr).replace(token, '***')[-1500:])
    if result.returncode != 0:
        print(f'WARNING: push of {stage} failed; the results are on Drive in', sources)

In [ ]:
# The raw ZuCo files are read many times; a local copy is much faster than Drive.
LOCAL_MAT = '/content/zuco_og_raw'
!mkdir -p $LOCAL_MAT && rsync -a --include='results*_SR.mat' --exclude='*' "$MAT_DIR/" "$LOCAL_MAT/"
!python -m pytest -q tests/test_frp.py tests/test_followup.py tests/test_encoding_scan.py

## B. Decoding the read word from EEG

A ridge map from each word's EEG to the word's identity vector (LaBSE input layer) is learned on training sentences and tested on unseen ones: 2-vs-2 (chance 50%), the same on pairs matched for length and frequency, retrieval among the test vocabulary, and the sentiment of the decoded word sequence. Baselines: shuffled EEG, noise, word features alone. No teacher forcing.

In [ ]:
OUT_B = f'{RESULTS}/decoding'
!python scripts/decode_eeg_to_text.py --dataset zuco --word-eeg-dir "$WORD_EEG_DIR" --out-dir "$OUT_B" \
    --vector-cache-dir "$VECTOR_CACHE" --device $DEVICE
!python scripts/decode_eeg_to_text.py --dataset teco --trt-dir "$TECO_TRT_DIR" --labels-csv "$TECO_LABELS" \
    --out-dir "$OUT_B" --vector-cache-dir "$VECTOR_CACHE" --device $DEVICE
push('decoding', OUT_B)

## C. English vs Persian EEG directions

In [ ]:
OUT_C = f'{RESULTS}/crosslingual'
!python scripts/crosslingual_subspaces.py --zuco-word-eeg-dir "$WORD_EEG_DIR" --teco-trt-dir "$TECO_TRT_DIR" \
    --teco-labels-csv "$TECO_LABELS" --out-dir "$OUT_C" --vector-cache-dir "$VECTOR_CACHE" --device $DEVICE
push('crosslingual', OUT_C)

## D–E. Fixation-related potentials (N400)

ZuCo stores the EEG of every fixation; each one is located inside the sentence EEG to get its onset, then the EEG from −100 to +700 ms around the word's first fixation is averaged in time windows (including the N400 window, 300–500 ms). **D** checks the file layout on 5 sentences and pushes the printout; **E** extracts all readers (resumable — re-run the cell if Colab disconnects).

In [ ]:
OUT_D = f'{RESULTS}/frp_probe'
os.makedirs(OUT_D, exist_ok=True)
!python scripts/extract_frp.py --mat-dir "$LOCAL_MAT" --labels-csv "$LABELS_CSV" --out-dir "$FRP_DIR" --probe 5 2>&1 | tee "$OUT_D/probe.txt"
push('frp_probe', OUT_D)

In [ ]:
!python scripts/extract_frp.py --mat-dir "$LOCAL_MAT" --labels-csv "$LABELS_CSV" --out-dir "$FRP_DIR"
OUT_E = f'{RESULTS}/frp_extraction'
os.makedirs(OUT_E, exist_ok=True)
!cp "$FRP_DIR"/frp_*.json "$OUT_E"/ 2>/dev/null; ls "$OUT_E"
push('frp_extraction', OUT_E)

## F. Fixation-related potentials: analyses

1. timing check (occipital lambda/P1 at ~100 ms) and the N400 regression (surprisal as positive control; valence and |valence| after lexical and timing controls);
2. the stage-1 signal probes on the fixation-locked features (positive controls and sentiment targets);
3. the encoding scan with word-feature, reading and **surprisal** controls;
4. decoding the read word from the fixation-locked EEG.

In [ ]:
OUT_F = f'{RESULTS}/frp_analysis'
WINDOWS = '0-100ms,100-200ms,200-300ms,300-400ms,400-500ms,500-600ms,600-700ms,N400_300-500ms'
!python scripts/analyze_frp.py --frp-dir "$FRP_DIR" --out-dir "$OUT_F/n400" --device $DEVICE
!python scripts/analyze_eeg_signal.py --word-eeg-dir "$FRP_DIR" --labels-csv "$LABELS_CSV" --out-dir "$OUT_F/signal" \
    --device $DEVICE --band-names "$WINDOWS"
!python scripts/scan_eeg_encoding.py --dataset zuco --tag zuco_frp --word-eeg-dir "$FRP_DIR" --out-dir "$OUT_F" \
    --vector-cache-dir "$VECTOR_CACHE" --surprisal-model gpt2 --device $DEVICE
!python scripts/decode_eeg_to_text.py --dataset zuco --tag zuco_frp --word-eeg-dir "$FRP_DIR" --out-dir "$OUT_F/decoding" \
    --vector-cache-dir "$VECTOR_CACHE" --device $DEVICE
push('frp_analysis', OUT_F)

## G. Brain-tuning the LoRA LLM (within-sentence EEG targets, 1 fold)

In [ ]:
OUT_G = f'{RESULTS}/brain_tuning'
!python scripts/run_brain_tuning.py --word-eeg-dir "$WORD_EEG_DIR" --results-dir "$OUT_G" \
    --run-tag brain_tuning_v2_centered --folds 1 --targets centered
push('brain_tuning', OUT_G)

## H. Optional: all five folds of the LLM with EEG tokens (many hours; finished folds are reused)

In [ ]:
RUN_TAG = 'v1_word_aligned'
!python scripts/run_eeg_text_lora.py --word-eeg-dir "$WORD_EEG_DIR" --results-dir "$LORA_RESULTS" --run-tag $RUN_TAG --folds 5
push('lora_5fold', f'{LORA_RESULTS}/{RUN_TAG}')